In [ ]:
# 불량 선별 실적 시계열 분석
- 프로젝트명: Defect-Sorting-Performance-Analysis
- 분석 기간: 2025년 8월 ~ 2026년 3월
- 분석 목적: 선별 실적 데이터에서 유의미한 추이/패턴을 도출한다.

## 분석 질문
1. 처리 수량은 시간의 흐름에 따라 어떤 추세를 보이는가? 
2. 처리 수량은 품종별로 특정 패턴이 있는가?
3. 처리 대상 불량의 유형별 구성비는 기간에 따라 어떤 차이가 있는가?

In [2]:
# 기본 라이브러리 import, 그래프 크기/스타일 미리 설정

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams["figure.figsize"] = (10, 6)
sns.set_style("whitegrid")

In [5]:
# 원본 파일 불러오기 

df = pd.read_excel("./data/RAW_data.xlsx")
df.head()

,구분,반,작업일,품종,Lot 코드,불량유형,처리 수량(BOX)
0,8월 1주차,오전,25.08.04,B,K5KKK1,홀 이상,4.0
1,8월 1주차,오전,25.08.05,B,K5KKK1,홀 이상,4.0
2,8월 1주차,오전,25.08.05,B,K5KKK2,홀 이상,2.0
3,8월 1주차,오후,25.08.05,B,K5KKK2,홀 이상,3.0
4,8월 1주차,오전,25.08.06,B,K5KKK2,홀 이상,3.0


In [6]:
# 시트 목록 확인

import pandas as pd

file_path = "./data/RAW_data.xlsx"

xls = pd.ExcelFile(file_path)
xls.sheet_names

['선별실적', '인원관리']

In [7]:
# 데이터 불러오기

df_perf = pd.read_excel(file_path, sheet_name="선별실적")
df_worker = pd.read_excel(file_path, sheet_name="인원관리")

c:\Users\user\AppData\Local\Programs\Python\Python314\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Conditional Formatting extension is not supported and will be removed
  warn(msg)


In [8]:
# 데이터 확인

display(df_perf.head())
display(df_worker.head())

print(df_perf.shape)
print(df_worker.shape)

,구분,반,작업일,품종,Lot 코드,불량유형,처리 수량(BOX)
0,8월 1주차,오전,25.08.04,B,K5KKK1,홀 이상,4.0
1,8월 1주차,오전,25.08.05,B,K5KKK1,홀 이상,4.0
2,8월 1주차,오전,25.08.05,B,K5KKK2,홀 이상,2.0
3,8월 1주차,오후,25.08.05,B,K5KKK2,홀 이상,3.0
4,8월 1주차,오전,25.08.06,B,K5KKK2,홀 이상,3.0


,작업일,반,인원수
0,25.08.04,오전,5.0
1,25.08.05,오전,6.0
2,25.08.05,오후,6.0
3,25.08.06,오전,6.0
4,25.08.06,오후,6.0


(420, 7)
(241, 3)


In [9]:
# 칼럼명 확인 및 정리

print(df_perf.columns.tolist())
print(df_worker.columns.tolist())

df_perf.columns = df_perf.columns.str.strip()
df_worker.columns = df_worker.columns.str.strip()

['구분', '반', '작업일', '품종', 'Lot 코드', '불량유형', '처리 수량(BOX)']
['작업일', '반', '인원수']


In [10]:
# 날짜형 변환

df_perf["작업일"] = pd.to_datetime(df_perf["작업일"], format="%y.%m.%d", errors="coerce")
df_worker["작업일"] = pd.to_datetime(df_worker["작업일"], format="%y.%m.%d", errors="coerce")

display(df_perf[["작업일"]].head())
display(df_worker[["작업일"]].head())

,작업일
0,2025-08-04
1,2025-08-05
2,2025-08-05
3,2025-08-05
4,2025-08-06


,작업일
0,2025-08-04
1,2025-08-05
2,2025-08-05
3,2025-08-06
4,2025-08-06


In [11]:
# 수치형 변환

df_perf["처리 수량(BOX)"] = pd.to_numeric(df_perf["처리 수량(BOX)"], errors="coerce")
df_worker["인원수"] = pd.to_numeric(df_worker["인원수"], errors="coerce")

In [12]:
# 결측치 확인

print(df_perf.isnull().sum())
print(df_worker.isnull().sum())

구분            0
반             0
작업일           0
품종            0
Lot 코드        0
불량유형          0
처리 수량(BOX)    0
dtype: int64
작업일    0
반      0
인원수    0
dtype: int64


In [13]:
# 중복 확인 및 제거

print("실적 데이터 중복 수:", df_perf.duplicated().sum())
print("인원수 데이터 중복 수:", df_worker.duplicated().sum())

df_perf = df_perf.drop_duplicates()
df_worker = df_worker.drop_duplicates()

실적 데이터 중복 수: 0
인원수 데이터 중복 수: 2


In [14]:
# 날짜형 통일
df_perf["작업일"] = pd.to_datetime(df_perf["작업일"]).dt.normalize()
df_worker["작업일"] = pd.to_datetime(df_worker["작업일"]).dt.normalize()

# df_perf의 작업일-구분 기준표
date_map = df_perf[["작업일", "구분"]].drop_duplicates()

# df_worker에 구분 붙이기
df_worker = df_worker.merge(date_map, on="작업일", how="left")

# 파생변수 생성
df_perf["연도"] = df_perf["작업일"].dt.year
df_perf["월"] = df_perf["구분"].str.extract(r"(\d+)월").astype(int)
df_perf["주차"] = df_perf["구분"].str.extract(r"(\d+)주차").astype(int)

df_worker["연도"] = df_worker["작업일"].dt.year
df_worker["월"] = df_worker["구분"].str.extract(r"(\d+)월").astype(int)
df_worker["주차"] = df_worker["구분"].str.extract(r"(\d+)주차").astype(int)

# 확인
print(df_perf[["작업일", "구분", "연도", "월", "주차"]].head())
print(df_worker[["작업일", "구분", "연도", "월", "주차"]].head())
print("df_worker 구분 결측치 수:", df_worker["구분"].isna().sum())

         작업일      구분    연도  월  주차
0 2025-08-04  8월 1주차  2025  8   1
1 2025-08-05  8월 1주차  2025  8   1
2 2025-08-05  8월 1주차  2025  8   1
3 2025-08-05  8월 1주차  2025  8   1
4 2025-08-06  8월 1주차  2025  8   1
         작업일      구분    연도  월  주차
0 2025-08-04  8월 1주차  2025  8   1
1 2025-08-05  8월 1주차  2025  8   1
2 2025-08-05  8월 1주차  2025  8   1
3 2025-08-06  8월 1주차  2025  8   1
4 2025-08-06  8월 1주차  2025  8   1
df_worker 구분 결측치 수: 0


In [15]:
# 인원수 데이터 결합(작업일+반 기준 합치기)

df = pd.merge(
    df_perf,
    df_worker[["작업일", "반", "인원수"]],
    on=["작업일", "반"],
    how="left"
)

display(df.head())
print(df.shape)

,구분,반,작업일,품종,Lot 코드,불량유형,처리 수량(BOX),연도,월,주차,인원수
0,8월 1주차,오전,2025-08-04,B,K5KKK1,홀 이상,4.0,2025,8,1,5.0
1,8월 1주차,오전,2025-08-05,B,K5KKK1,홀 이상,4.0,2025,8,1,6.0
2,8월 1주차,오전,2025-08-05,B,K5KKK2,홀 이상,2.0,2025,8,1,6.0
3,8월 1주차,오후,2025-08-05,B,K5KKK2,홀 이상,3.0,2025,8,1,6.0
4,8월 1주차,오전,2025-08-06,B,K5KKK2,홀 이상,3.0,2025,8,1,6.0


(420, 11)


In [ ]:
# 작업일 + 반 조합으로 인원수 합치기(merge)

df = pd.merge(
    df_perf,
    df_worker[["작업일", "반", "인원수"]],
    on=["작업일", "반"],
    how="left"
)

In [18]:
# merge 결과 확인

df.head()

print("최종 데이터 크기:", df.shape)

최종 데이터 크기: (420, 11)


In [19]:
# 최종 결측치 확인

df.isnull().sum()

구분            0
반             0
작업일           0
품종            0
Lot 코드        0
불량유형          0
처리 수량(BOX)    0
연도            0
월             0
주차            0
인원수           4
dtype: int64

In [ ]:
# 데이터 정보 확인(각 컬럼 타입/null 개수/전체 행 개수)

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 420 entries, 0 to 419
Data columns (total 11 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   구분          420 non-null    str           
 1   반           420 non-null    str           
 2   작업일         420 non-null    datetime64[us]
 3   품종          420 non-null    str           
 4   Lot 코드      420 non-null    str           
 5   불량유형        420 non-null    str           
 6   처리 수량(BOX)  420 non-null    float64       
 7   연도          420 non-null    int32         
 8   월           420 non-null    int64         
 9   주차          420 non-null    int64         
 10  인원수         416 non-null    float64       
dtypes: datetime64[us](1), float64(2), int32(1), int64(2), str(5)
memory usage: 34.6 KB
